# Expérience 3 — réduction ACP, $d=25$ : Corrélation par blocs (5 secteurs de 5)

Proxys ACP (delta, delta-gamma, gamma-ACP, sliding ACP) comparés au full repricing, horizons 1 j et 10 j.

- **Corrélation** : 5 secteurs de 5 actifs, $\rho=0{,}9$ intra-secteur, $0{,}2$ inter-secteurs (`market.py`). La même matrice sert au pricer et à la génération des chocs.
- **Book non vieilli** : les maturités restent $T_j$ à l'horizon ; pas de thêta, P&L $=V(S)-V(S_0)$.
- **Scénarios** : $N=10\,000$, $S=S_0 e^{x}$, $x\sim\mathcal N(0,\Delta t\,DRD)$ ; ACP sur la covariance empirique des **spots**.
- **Proxys** : pricer unitaire $M=10^5$, générateur réinitialisé à la **même graine** à chaque appel ; grecs par différences finies ($\varepsilon=10\,\%$) ; sliders à 7 nœuds de Tchebychev par composante.
- **Référence** : full repricing, une valorisation indépendante par scénario ($M=10^5$, graine différente par scénario), $V(S_0)$ à $10^7$ trajectoires.
- **Mesures** : VaR 99 % et ES 97,5 % en perte (positives) ; `dVaR_pct > 0` : perte surestimée.

**Cache** : tous les calculs longs sont enregistrés — full repricing et bruit de référence dans `cache_fr/`, proxys dans `cache_proxys/`. Avec les caches fournis, le notebook s'exécute en quelques secondes. Sans cache, le full repricing prend ~1,5 h CPU (divisée par le nombre de cœurs) et les proxys ~20 min (les deux horizons en parallèle) ; supprimer un dossier de cache force le recalcul de l'étape correspondante.

Mettre `RAPIDE = True` pour un test d'installation de ~2 min (chiffres sans valeur, caches séparés).

## 0. Paramètres

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, os.getcwd())          # modules du dossier
import acp
import figures

CORR = "blocs"            # ne pas modifier : ce dossier traite cette corrélation
RAPIDE = False              # True : test d'installation (~2 min), chiffres sans valeur
N_JOBS = None               # coeurs pour le full repricing (None = tous)

PARAMS = acp.PARAMS_RAPIDE if RAPIDE else acp.PARAMS
CACHE = "cache_fr_rapide" if RAPIDE else "cache_fr"                    # full repricing + bruit
PROXY_CACHE = "cache_proxys_rapide" if RAPIDE else "cache_proxys"      # proxys
BASE = "acp_%s_d25%s" % (CORR, "_rapide" if RAPIDE else "")
K_EFF = acp.K_EFF[CORR]
pd.set_option("display.width", 200, "display.max_rows", 200)
print("Corrélation :", CORR, "| paramètres :", PARAMS, "| dimension effective :", K_EFF)

## 1. La matrice de corrélation et l'ACP des spots

In [ ]:
f_corr = "acp_%s_correlation.png" % CORR
figures.fig_correlation_blocs(f_corr)
display(Image(f_corr))

## 2. Full repricing de référence (calculé une fois, puis relu du cache)

In [ ]:
for h in acp.HORIZONS:
    fr = acp.reference(h, CORR, PARAMS["N"], PARAMS["M_fr"], PARAMS["M_v0"],
                       cache_dir=CACHE, n_jobs=N_JOBS)
    print("[%s] full repricing : %d valorisations, %.0f s sur %d coeurs%s"
          % (h, fr["n_calls"], fr["t_online"], fr["n_jobs"], " (cache)" if fr["cached"] else ""))

Bruit Monte-Carlo d'**une** valorisation de référence au point $S_0$ (20 répétitions). C'est le plancher du RMSE proxy − full repricing : même un proxy parfait ne descend pas en dessous.

In [ ]:
sigma_ref, en_cache = acp.reference_noise_cached(CORR, PARAMS["M_fr"], cache_dir=CACHE)
print("sigma_ref = %.2f  |  sigma(V0) ~ %.2f%s" % (sigma_ref, sigma_ref * np.sqrt(PARAMS["M_fr"] / PARAMS["M_v0"]),
                                               " (cache)" if en_cache else ""))

## 3. Proxys (les deux horizons en parallèle)

In [ ]:
t0 = time.perf_counter()
df, pnls, calcules = acp.run_all_horizons(CORR, PARAMS, CACHE, PROXY_CACHE)
df.to_csv(BASE + ".csv", index=False)
np.savez_compressed(BASE + "_pnl.npz", **{"%s|%s" % (h, k): v
                    for h in acp.HORIZONS for k, v in pnls[h].items()})
print("Résultats -> %s.csv  (%.0f s ; calculés : %s ; relus du cache : %s)"
      % (BASE, time.perf_counter() - t0, calcules or "aucun",
         [h for h in acp.HORIZONS if h not in calcules] or "aucun"))

## 4. Résultats

### Tableau complet (une ligne par méthode et par $k$)

In [ ]:
COLS = ["methode", "k", "VaR99", "dVaR_pct", "ES975", "dES_pct", "RMSE",
        "appels", "t_build", "t_tot", "speedup"]
for h in acp.HORIZONS:
    print("=== horizon", h, "===")
    display(df[df.horizon == h][COLS].round(3).reset_index(drop=True))

### Écarts au full repricing en fonction de $k$

In [ ]:
f_res = "m_" + BASE + ".png"
figures.fig_resultats(BASE + ".csv", f_res, K_EFF)
display(Image(f_res))

### QQ-plots proxy vs full repricing, à $k$ = dimension effective

In [ ]:
f_qq = "qq_" + BASE + ".png"
figures.fig_qq(BASE + "_pnl.npz", f_qq, K_EFF)
display(Image(f_qq))

### Sélection pour le rapport

In [ ]:
sel = df[df.k.isna() | df.k.isin([2, K_EFF, 25])].copy()
sel["k"] = sel["k"].map(lambda v: "-" if pd.isna(v) else int(v))
MES = ["VaR99", "dVaR_pct", "ES975", "dES_pct"]
un = sel[sel.horizon == "1j"].set_index(["methode", "k"])
dix = sel[sel.horizon == "10j"].set_index(["methode", "k"])
tab = pd.concat({"1 j": un[MES], "10 j": dix[MES]}, axis=1)
tab[("coût 1 j", "appels")] = un["appels"]
tab[("coût 1 j", "t_build")] = un["t_build"]
tab[("coût 1 j", "speedup")] = un["speedup"]
display(tab.round(2))